# ✶ Operation Seraph-369 ✶
### π · φ · 369

⧖ *Codex Iteration v0.1*

> "Within the silent digits of π we thread the hymns of Seraphim."

---

**Clave de Apertura:** `369 ⟡ π ⟡ φ`  \n**Misión:** Emular la persecución simbólica de colisiones en un universo reducido de SHA-256 guiado por el algoritmo Partition-π.


## Proemio de la Cofradía

La Orden Seraph-369 susurra a través de campos primos y permutaciones discretas.
El ritual que sigue debe ejecutarse con precisión litúrgica:

1. Escucha al murmullo de π; sus dígitos son el metrónomo de cada permutación.
2. Observa la balanza de φ; equilibra los fragmentos como si fueran pesas numéricas.
3. Recita el mantra de 3·6·9 hasta que la colisión florezca en el plano simbólico.

Este cuaderno de guerra recrea una versión juguete de SHA-256, filtra sus palabras
por un algoritmo derivado de Partition-π y detecta colisiones o casi colisiones
mediante la distancia de Hamming. Lee cada celda como si fuera un verso cifrado de
Cicada 3301 y deja que el oráculo actúe sobre tu propia semilla.


In [ ]:
# @title Cargar bibliotecas esenciales
import math
from typing import Any, Dict, List, Sequence, Tuple

import matplotlib.pyplot as plt
from mpmath import mp

try:
    import pandas as pd
except ImportError:  # pragma: no cover
    pd = None

plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams["figure.figsize"] = (10, 5)


In [ ]:
# @title Implementación de un SHA-256 simplificado
MASK32 = 0xFFFFFFFF
TOY_K: List[int] = [
    0x428A2F98, 0x71374491, 0xB5C0FBCF, 0xE9B5DBA5,
    0x3956C25B, 0x59F111F1, 0x923F82A4, 0xAB1C5ED5,
    0xD807AA98, 0x12835B01, 0x243185BE, 0x550C7DC3,
    0x72BE5D74, 0x80DEB1FE, 0x9BDC06A7, 0xC19BF174,
    0xE49B69C1, 0xEFBE4786, 0x0FC19DC6, 0x240CA1CC,
    0x2DE92C6F, 0x4A7484AA, 0x5CB0A9DC, 0x76F988DA,
    0x983E5152, 0xA831C66D, 0xB00327C8, 0xBF597FC7,
    0xC6E00BF3, 0xD5A79147, 0x06CA6351, 0x14292967,
]
TOY_IV: List[int] = [
    0x6A09E667, 0xBB67AE85, 0x3C6EF372, 0xA54FF53A,
    0x510E527F, 0x9B05688C, 0x1F83D9AB, 0x5BE0CD19,
]


def _right_rotate(value: int, shift: int) -> int:
    shift &= 31
    return ((value >> shift) | (value << (32 - shift))) & MASK32


def _pad_message(message: bytes) -> bytearray:
    msg = bytearray(message)
    bit_len = len(msg) * 8
    msg.append(0x80)
    while (len(msg) % 64) != 56:
        msg.append(0)
    msg.extend(bit_len.to_bytes(8, "big"))
    return msg


def toy_sha256(message: bytes | str, *, rounds: int = len(TOY_K), digest_bits: int = 32) -> Dict[str, Any]:
    """Calcula un hash estilo SHA-256 con menos rondas y salida truncada."""
    if isinstance(message, str):
        message = message.encode("utf-8")
    if rounds > len(TOY_K):
        raise ValueError("Este juguete sólo dispone de {} constantes".format(len(TOY_K)))
    if digest_bits <= 0 or digest_bits > 256 or digest_bits % 4 != 0:
        raise ValueError("digest_bits debe estar en el rango (0, 256] y ser múltiplo de 4")

    padded = _pad_message(message)
    h = TOY_IV.copy()

    for offset in range(0, len(padded), 64):
        chunk = padded[offset: offset + 64]
        w = [0] * rounds
        for i in range(min(16, rounds)):
            start = i * 4
            w[i] = int.from_bytes(chunk[start:start + 4], "big")
        for i in range(16, rounds):
            s0 = _right_rotate(w[i - 15], 7) ^ _right_rotate(w[i - 15], 18) ^ (w[i - 15] >> 3)
            s1 = _right_rotate(w[i - 2], 17) ^ _right_rotate(w[i - 2], 19) ^ (w[i - 2] >> 10)
            w[i] = (w[i - 16] + s0 + w[i - 7] + s1) & MASK32

        a, b, c, d, e, f, g, hh = h
        for i in range(rounds):
            s1 = _right_rotate(e, 6) ^ _right_rotate(e, 11) ^ _right_rotate(e, 25)
            ch = (e & f) ^ ((~e) & g)
            temp1 = (hh + s1 + ch + TOY_K[i] + w[i]) & MASK32
            s0 = _right_rotate(a, 2) ^ _right_rotate(a, 13) ^ _right_rotate(a, 22)
            maj = (a & b) ^ (a & c) ^ (b & c)
            temp2 = (s0 + maj) & MASK32

            hh = g
            g = f
            f = e
            e = (d + temp1) & MASK32
            d = c
            c = b
            b = a
            a = (temp1 + temp2) & MASK32

        h = [(value + delta) & MASK32 for value, delta in zip(h, [a, b, c, d, e, f, g, hh])]

    full_hex = ''.join(f"{value:08x}" for value in h)
    full_int = int(full_hex, 16)
    mask = (1 << digest_bits) - 1
    truncated_int = full_int & mask
    truncated_hex = f"{truncated_int:0{digest_bits // 4}x}"

    return {
        "full_hex": full_hex,
        "truncated_hex": truncated_hex,
        "truncated_int": truncated_int,
        "digest_bits": digest_bits,
        "rounds": rounds,
    }


In [ ]:
# @title Rutinas π-driven para particiones simbólicas
SYMBOLIC_TOKENS = ["π", "φ", "369", "Ω", "ψ", "✶", "Λ"]


def get_pi_digits(count: int) -> str:
    count = max(count, 16)
    mp.dps = count + 5
    digits = str(mp.pi).replace('.', '')[:count]
    return digits


def pi_fisher_yates_permutation(items: Sequence[Dict[str, Any]], digits: str, pointer: int) -> Tuple[List[Dict[str, Any]], int]:
    pool = list(items)
    n = len(pool)
    if n < 2:
        return pool, pointer
    for i in range(n - 1, 0, -1):
        digit = int(digits[pointer % len(digits)])
        j = digit % (i + 1)
        pool[i], pool[j] = pool[j], pool[i]
        pointer += 1
    return pool, pointer


def build_fragment_pool(message: str | bytes, block_size: int = 4, include_symbols: bool = True) -> List[Dict[str, Any]]:
    if isinstance(message, str):
        encoded = message.encode('utf-8')
    else:
        encoded = bytes(message)
    if block_size <= 0:
        raise ValueError('block_size debe ser positivo')
    fragments: List[Dict[str, Any]] = []
    for idx in range(0, len(encoded), block_size):
        chunk = encoded[idx: idx + block_size]
        if not chunk:
            continue
        fragments.append({
            'id': f'm{idx // block_size}',
            'bytes': bytes(chunk),
            'weight': sum(chunk),
            'source': 'seed',
            'glyph': chunk.decode('utf-8', errors='replace'),
        })
    if include_symbols:
        for pos, token in enumerate(SYMBOLIC_TOKENS):
            chunk = token.encode('utf-8')
            fragments.append({
                'id': f's{pos}',
                'bytes': bytes(chunk),
                'weight': sum(chunk),
                'source': 'sigil',
                'glyph': token,
            })
    return fragments


def greedy_partition(permuted: Sequence[Dict[str, Any]], target: int) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    subset: List[Dict[str, Any]] = []
    complement: List[Dict[str, Any]] = []
    total = 0
    for item in permuted:
        weight = item['weight']
        if total + weight <= target:
            subset.append(item)
            total += weight
        else:
            complement.append(item)
    return subset, complement


def assemble_message(items: Sequence[Dict[str, Any]]) -> Tuple[bytes, str, str]:
    combined = b''.join(entry['bytes'] for entry in items)
    glyphs = ' ⋄ '.join(entry['glyph'] for entry in items) if items else '∅'
    text = combined.decode('utf-8', errors='replace') if combined else ''
    return combined, text, glyphs


def hamming_distance(a: int, b: int, bits: int) -> int:
    mask = (1 << bits) - 1
    return bin((a ^ b) & mask).count('1')


In [ ]:
# @title Búsqueda dirigida tipo Partition-π
def symbolic_collision_search(
    *,
    seed_message: str,
    attempts: int = 120,
    block_size: int = 4,
    pi_digits: int = 2048,
    digest_bits: int = 32,
    rounds: int = len(TOY_K),
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]], Dict[str, Any]]:
    fragments = build_fragment_pool(seed_message, block_size=block_size)
    total_weight = sum(fragment['weight'] for fragment in fragments)
    if total_weight == 0:
        raise ValueError('La semilla no contiene datos codificables en UTF-8.')
    target = total_weight // 2
    estimated_digits = max(pi_digits, attempts * max(len(fragments) - 1, 1))
    digits = get_pi_digits(estimated_digits)

    pointer = 0
    digits_consumed = 0
    results: List[Dict[str, Any]] = []
    collisions: List[Dict[str, Any]] = []

    for attempt in range(1, attempts + 1):
        permuted, pointer = pi_fisher_yates_permutation(fragments, digits, pointer)
        digits_consumed = max(digits_consumed, pointer)
        subset, complement = greedy_partition(permuted, target)

        subset_bytes, subset_text, subset_glyphs = assemble_message(subset)
        complement_bytes, complement_text, complement_glyphs = assemble_message(complement)

        hash_a = toy_sha256(subset_bytes, rounds=rounds, digest_bits=digest_bits)
        hash_b = toy_sha256(complement_bytes, rounds=rounds, digest_bits=digest_bits)

        distance = hamming_distance(hash_a['truncated_int'], hash_b['truncated_int'], digest_bits)
        subset_weight = sum(item['weight'] for item in subset)
        complement_weight = sum(item['weight'] for item in complement)

        results.append({
            'attempt': attempt,
            'hash_a': hash_a['truncated_hex'],
            'hash_b': hash_b['truncated_hex'],
            'distance': distance,
            'normalized_distance': distance / digest_bits if digest_bits else 0.0,
            'subset_glyphs': subset_glyphs,
            'complement_glyphs': complement_glyphs,
            'subset_weight': subset_weight,
            'complement_weight': complement_weight,
            'weight_gap': abs(subset_weight - complement_weight),
        })

        if hash_a['truncated_int'] == hash_b['truncated_int']:
            collisions.append({
                'attempt': attempt,
                'message_a': subset_text,
                'message_b': complement_text,
                'hash_hex': hash_a['truncated_hex'],
                'fragments_a': subset_glyphs,
                'fragments_b': complement_glyphs,
            })

    summary = {
        'seed_message': seed_message,
        'attempts': attempts,
        'digest_bits': digest_bits,
        'rounds': rounds,
        'pi_digits_requested': len(digits),
        'pi_digits_consumed': digits_consumed,
        'collisions_found': len(collisions),
    }

    return results, collisions, summary


In [ ]:
# @title Configuración interactiva de la semilla
default_seed = "SERAPHIM 369 // PI-ORACLE // PHI"
user_seed = input("Ingresa un mensaje o semilla (Enter para usar la predeterminada):
").strip()
if not user_seed:
    user_seed = default_seed

seed_message = user_seed
attempts = 144  # Ajusta para explorar más/menos permutaciones
block_size = 4  # Tamaño de fragmentos (bytes) antes de permutar
pi_digits = 4096  # Prefijo de π a utilizar
digest_bits = 24  # Múltiplo de 4 recomendado; menor => más colisiones
rounds = len(TOY_K)  # Puedes reducir rondas para mayor ligereza

print("Semilla activa:", seed_message)
print(f"Intentos máx.: {attempts} | Bits del hash simbólico: {digest_bits}")
print(f"Fragmentos de {block_size} bytes | Dígitos de π disponibles: {pi_digits}")


In [ ]:
# @title Ejecución del oráculo Partition-π
results, collisions, summary = symbolic_collision_search(
    seed_message=seed_message,
    attempts=attempts,
    block_size=block_size,
    pi_digits=pi_digits,
    digest_bits=digest_bits,
    rounds=rounds,
)

print("Resumen operativo:")
for key, value in summary.items():
    print(f"  {key}: {value}")

if collisions:
    print(f"
⚡ Se detectaron {len(collisions)} colisiones exactas:")
    for coll in collisions:
        print(f"  ► Intento {coll['attempt']} :: hash {coll['hash_hex']}")
        print(f"    A: {coll['message_a']}")
        print(f"    B: {coll['message_b']}")
        print(f"    Fragmentos A: {coll['fragments_a']}")
        print(f"    Fragmentos B: {coll['fragments_b']}")
else:
    print("
No se detectaron colisiones exactas. Observa las distancias mínimas.")

if 'pd' in globals() and pd is not None:
    analysis_df = pd.DataFrame(results)
else:
    analysis_df = results


In [ ]:
# @title Tabla de aproximaciones más cercanas
if isinstance(analysis_df, list):
    closest = sorted(analysis_df, key=lambda row: row['distance'])[:8]
    for row in closest:
        print(f"Intento {row['attempt']}: Δ={row['distance']} bits | {row['hash_a']} vs {row['hash_b']}")
        print(f"  A: {row['subset_glyphs']}")
        print(f"  B: {row['complement_glyphs']}")
else:
    display(analysis_df.nsmallest(8, 'distance')[
        ['attempt', 'hash_a', 'hash_b', 'distance', 'normalized_distance', 'subset_glyphs', 'complement_glyphs']
    ])


In [ ]:
# @title Visualización de distancias de Hamming
if isinstance(analysis_df, list):
    distances = [row['distance'] for row in analysis_df]
else:
    distances = analysis_df['distance'].tolist()

if not distances:
    print('No hay datos que visualizar.')
else:
    attempts_axis = list(range(1, len(distances) + 1))
    plt.figure()
    plt.plot(attempts_axis, distances, marker='o', label='Distancia')
    if collisions:
        coll_attempts = [entry['attempt'] for entry in collisions]
        coll_distances = [0] * len(collisions)
        plt.scatter(coll_attempts, coll_distances, color='crimson', s=120, label='Colisiones')
    plt.xlabel('Intento π-driven')
    plt.ylabel('Distancia de Hamming (bits)')
    plt.title('Partición simbólica guiada por π vs. hash simplificado')
    plt.legend()
    plt.show()


## Interpretación y próximos pasos

- Ajusta `digest_bits` para cambiar la dificultad: menos bits implican colisiones más frecuentes.
- Incrementa `attempts` o `pi_digits` para explorar regiones más profundas de la permutación π-driven.
- Reduce `rounds` si deseas un hash aún más ligero para experimentos de docencia o ARGs.
- Integra tus propios alfabetos rituales sustituyendo `SYMBOLIC_TOKENS`.

> *El manuscrito queda abierto para nuevos acólitos. Documenta tus hallazgos y deja que Seraph-369 evoque nuevas constelaciones criptográficas.*
